# Redes Neuronales Básicas desde Cero (Vainilla)
### Fundamentos de Machine Learning y Deep Learning con Python y Numpy

**Autor:** Daniel Fernando Contreras Abella

En este notebook se implementan, **sin usar librerías especializadas de Deep Learning** (como TensorFlow, PyTorch o Keras), los siguientes modelos:

1. Un **Perceptrón** simple.
2. Una **Red Neuronal de una capa** (una sola capa de neuronas, entrenada con descenso de gradiente).
3. Una **Red Neuronal Multicapa (MLP - Vainilla)** con retropropagación (backpropagation).

Todas las operaciones matemáticas (productos punto, sumas matriciales, funciones de activación) se implementan utilizando **Numpy**, aprovechando la **vectorización** para procesar varios ejemplos de datos a la vez, en lugar de recorrerlos uno por uno con ciclos explícitos.

---

## 1. Conceptos fundamentales

- **Machine Learning (ML):** rama de la Inteligencia Artificial en la que un modelo aprende patrones a partir de datos, ajustando parámetros internos (pesos) para minimizar un error, en lugar de ser programado explícitamente con reglas fijas.
- **Deep Learning (DL):** subcampo del ML que utiliza redes neuronales artificiales con una o varias capas ocultas para aprender representaciones jerárquicas de los datos.
- **Perceptrón:** la unidad más simple de una red neuronal. Recibe varias entradas, las multiplica por unos pesos, suma un sesgo (bias) y aplica una función de activación para producir una salida binaria.
- **Red neuronal de una capa:** conjunto de una o varias neuronas organizadas en una única capa que reciben directamente las entradas y producen la(s) salida(s), sin capas intermedias.
- **Red neuronal multicapa (MLP):** incluye una o más **capas ocultas** entre la entrada y la salida, lo que le permite aprender relaciones no lineales que un perceptrón simple no puede resolver (por ejemplo, la función XOR).
- **Vectorización:** técnica que consiste en expresar operaciones sobre conjuntos completos de datos (matrices) usando operaciones matriciales de Numpy en lugar de ciclos `for` en Python puro. Esto hace el código más rápido y más legible.


In [ ]:
# Importamos Numpy, la única librería necesaria para las operaciones matemáticas y matriciales.
import numpy as np

# Fijamos una semilla para que los resultados aleatorios sean reproducibles.
np.random.seed(42)


## 2. Perceptrón Simple

El perceptrón calcula una combinación lineal de las entradas (`z = X . W + b`) y aplica una **función escalón** (step function) como activación:

- Si `z >= 0` → salida = 1
- Si `z < 0`  → salida = 0

El aprendizaje se realiza mediante la **regla del perceptrón**: los pesos se ajustan proporcionalmente al error cometido en cada ejemplo.

Vamos a entrenar un perceptrón para aprender la compuerta lógica **AND**.


In [ ]:
class Perceptron:
    """Perceptrón simple con función de activación escalón."""

    def __init__(self, n_entradas, tasa_aprendizaje=0.1):
        # Inicializamos los pesos en cero y el sesgo (bias) en cero.
        self.pesos = np.zeros(n_entradas)
        self.bias = 0.0
        self.lr = tasa_aprendizaje

    def activacion(self, z):
        # Función escalón: 1 si z >= 0, de lo contrario 0.
        return 1 if z >= 0 else 0

    def predecir(self, x):
        # Combinación lineal de entradas y pesos, más el sesgo.
        z = np.dot(x, self.pesos) + self.bias
        return self.activacion(z)

    def entrenar(self, X, y, epocas=10):
        # Ciclo de entrenamiento: recorremos varias veces (épocas) todo el dataset.
        for epoca in range(epocas):
            for xi, objetivo in zip(X, y):
                prediccion = self.predecir(xi)
                error = objetivo - prediccion
                # Regla de actualización del perceptrón.
                self.pesos += self.lr * error * xi
                self.bias += self.lr * error


# Datos predefinidos: tabla de verdad de la compuerta AND
X_and = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]])
y_and = np.array([0, 0, 0, 1])

perceptron = Perceptron(n_entradas=2, tasa_aprendizaje=0.1)
perceptron.entrenar(X_and, y_and, epocas=10)

print("Resultados del Perceptrón entrenado para la compuerta AND:")
for xi in X_and:
    print(f"Entrada: {xi} -> Predicción: {perceptron.predecir(xi)}")

print("\nPesos finales:", perceptron.pesos)
print("Bias final:", perceptron.bias)


Resultados del Perceptrón entrenado para la compuerta AND:
Entrada: [0 0] -> Predicción: 0
Entrada: [0 1] -> Predicción: 0
Entrada: [1 0] -> Predicción: 0
Entrada: [1 1] -> Predicción: 1

Pesos finales: [0.2 0.1]
Bias final: -0.20000000000000004


## 3. Red Neuronal de una Capa

A diferencia del perceptrón (que usa una función escalón no derivable), aquí utilizamos:

- La función de activación **sigmoide**: `sigmoid(z) = 1 / (1 + e^-z)`, que produce salidas continuas entre 0 y 1.
- **Descenso de gradiente** para actualizar los pesos, usando la derivada de la función sigmoide.

Toda la red (para los 4 ejemplos de entrenamiento a la vez) se procesa de forma **vectorizada**: en cada época se hace un solo producto matricial `X . W`, en lugar de recorrer cada ejemplo con un `for`.

Entrenamos esta red para que también aprenda la compuerta **AND**, pero ahora con una neurona de salida entrenada mediante gradiente.


In [ ]:
def sigmoid(z):
    """Función de activación sigmoide."""
    return 1 / (1 + np.exp(-z))

def sigmoid_derivada(a):
    """Derivada de la sigmoide, expresada en función de la salida 'a' ya activada."""
    return a * (1 - a)


# Datos predefinidos (compuerta AND)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
y = np.array([[0], [0], [0], [1]])

n_entradas = 2
n_salidas = 1

# Inicialización de pesos y bias de forma aleatoria (matriz de pesos: entradas x salidas)
W = np.random.randn(n_entradas, n_salidas)
b = np.zeros((1, n_salidas))

tasa_aprendizaje = 0.5
epocas = 10000

for epoca in range(epocas):
    # --- Forward pass (vectorizado para los 4 ejemplos a la vez) ---
    z = np.dot(X, W) + b
    a = sigmoid(z)

    # --- Cálculo del error ---
    error = y - a

    # --- Backward pass: gradiente respecto a los pesos y al bias ---
    delta = error * sigmoid_derivada(a)
    W += tasa_aprendizaje * np.dot(X.T, delta)
    b += tasa_aprendizaje * np.sum(delta, axis=0, keepdims=True)

    if epoca % 2000 == 0:
        perdida = np.mean(np.square(error))
        print(f"Época {epoca:5d} - Pérdida (MSE): {perdida:.6f}")

print("\nPredicciones finales de la red de una capa (compuerta AND):")
salida_final = sigmoid(np.dot(X, W) + b)
for xi, pred in zip(X, salida_final):
    print(f"Entrada: {xi} -> Salida: {pred[0]:.4f} -> Clase: {round(pred[0])}")


Época     0 - Pérdida (MSE): 0.255593
Época  2000 - Pérdida (MSE): 0.002642
Época  4000 - Pérdida (MSE): 0.001243
Época  6000 - Pérdida (MSE): 0.000806
Época  8000 - Pérdida (MSE): 0.000594

Predicciones finales de la red de una capa (compuerta AND):
Entrada: [0 0] -> Salida: 0.0000 -> Clase: 0
Entrada: [0 1] -> Salida: 0.0235 -> Clase: 0
Entrada: [1 0] -> Salida: 0.0235 -> Clase: 0
Entrada: [1 1] -> Salida: 0.9721 -> Clase: 1


## 4. Red Neuronal Multicapa (MLP - Vainilla)

Tanto el perceptrón como la red de una capa son modelos **lineales**: solo pueden separar clases mediante una línea recta (o hiperplano). Por eso **no pueden aprender la compuerta XOR**, que no es linealmente separable.

Para resolver este problema construimos una **red neuronal multicapa (Multi-Layer Perceptron)** con:

- Una **capa oculta** con varias neuronas y activación sigmoide.
- Una **capa de salida** con una neurona y activación sigmoide.
- Entrenamiento mediante **backpropagation** (retropropagación del error), que consiste en propagar el error de la capa de salida hacia la capa oculta usando la regla de la cadena, y así ajustar los pesos de ambas capas.

Todo el proceso, tanto el *forward pass* como el *backward pass*, se implementa con operaciones matriciales de Numpy (vectorización), procesando los 4 ejemplos de la tabla de verdad simultáneamente.


In [ ]:
# Datos predefinidos: tabla de verdad de la compuerta XOR (NO linealmente separable)
X_xor = np.array([[0, 0],
                   [0, 1],
                   [1, 0],
                   [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

# Arquitectura de la red: 2 entradas -> 4 neuronas ocultas -> 1 salida
n_entradas = 2
n_ocultas = 4
n_salidas = 1

# Inicialización aleatoria de pesos y bias para ambas capas
W1 = np.random.randn(n_entradas, n_ocultas)
b1 = np.zeros((1, n_ocultas))
W2 = np.random.randn(n_ocultas, n_salidas)
b2 = np.zeros((1, n_salidas))

tasa_aprendizaje = 0.5
epocas = 10000

for epoca in range(epocas):
    # ---------- FORWARD PASS ----------
    # Capa oculta
    z1 = np.dot(X_xor, W1) + b1
    a1 = sigmoid(z1)

    # Capa de salida
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)

    # ---------- BACKWARD PASS (backpropagation) ----------
    # Error en la capa de salida
    error_salida = y_xor - a2
    delta2 = error_salida * sigmoid_derivada(a2)

    # Error propagado hacia la capa oculta
    error_oculta = np.dot(delta2, W2.T)
    delta1 = error_oculta * sigmoid_derivada(a1)

    # ---------- ACTUALIZACIÓN DE PESOS Y BIAS ----------
    W2 += tasa_aprendizaje * np.dot(a1.T, delta2)
    b2 += tasa_aprendizaje * np.sum(delta2, axis=0, keepdims=True)

    W1 += tasa_aprendizaje * np.dot(X_xor.T, delta1)
    b1 += tasa_aprendizaje * np.sum(delta1, axis=0, keepdims=True)

    if epoca % 2000 == 0:
        perdida = np.mean(np.square(error_salida))
        print(f"Época {epoca:5d} - Pérdida (MSE): {perdida:.6f}")

# ---------- PREDICCIONES FINALES ----------
z1 = np.dot(X_xor, W1) + b1
a1 = sigmoid(z1)
z2 = np.dot(a1, W2) + b2
a2 = sigmoid(z2)

print("\nPredicciones finales de la red multicapa (compuerta XOR):")
for xi, pred in zip(X_xor, a2):
    print(f"Entrada: {xi} -> Salida: {pred[0]:.4f} -> Clase: {round(pred[0])}")


Época     0 - Pérdida (MSE): 0.363516
Época  2000 - Pérdida (MSE): 0.002012
Época  4000 - Pérdida (MSE): 0.000727
Época  6000 - Pérdida (MSE): 0.000434
Época  8000 - Pérdida (MSE): 0.000307

Predicciones finales de la red multicapa (compuerta XOR):
Entrada: [0 0] -> Salida: 0.0155 -> Clase: 0
Entrada: [0 1] -> Salida: 0.9854 -> Clase: 1
Entrada: [1 0] -> Salida: 0.9853 -> Clase: 1
Entrada: [1 1] -> Salida: 0.0166 -> Clase: 0


## 5. Comparación y Conclusiones

| Modelo | Activación | Aprendizaje | ¿Resuelve XOR? |
|---|---|---|---|
| Perceptrón simple | Escalón | Regla del perceptrón | No |
| Red neuronal de una capa | Sigmoide | Descenso de gradiente | No (sigue siendo lineal) |
| Red neuronal multicapa (MLP) | Sigmoide en cada capa | Backpropagation | **Sí** |

**Conclusiones:**

- El **perceptrón** y la **red de una capa** solo pueden resolver problemas **linealmente separables** (como AND, OR), porque matemáticamente solo generan un hiperplano de decisión.
- La **red multicapa** incorpora una **capa oculta**, lo que le permite combinar varias fronteras de decisión lineales y así aproximar funciones no lineales como XOR.
- La **vectorización con Numpy** permite procesar todos los ejemplos del dataset en cada iteración usando productos matriciales (`np.dot`), en lugar de ciclos `for` sobre cada ejemplo, lo cual es mucho más eficiente y es la base de cómo funcionan las librerías de Deep Learning "por dentro".
- Comprender estos modelos "desde cero" ayuda a entender qué hacen realmente librerías como TensorFlow o PyTorch cuando se llama a `model.fit()`.

---


**Enlace del repositorio GitHub:** https://github.com/danielcontreras205/Redes-Neuronales-b-sicas/blob/main/Redes_Neuronales_Basicas.ipynb

**Enlace de Google Colab:** https://colab.research.google.com/drive/1fyLvOgI7jj2c1lxU00qKgUU3mIqVLZge?usp=sharing
